# 09 — Modelos locales con Ollama

> **Bloque:** LLMs | **Nivel:** Intermedio
>
> Complementario al tutorial [09-modelos-locales-ollama.md](../../llms/09-modelos-locales-ollama.md)

Ejecutamos modelos de lenguaje completamente en local: sin API keys, sin costes por token, sin que los datos salgan de tu máquina.

**Prerequisito:** tener [Ollama instalado](https://ollama.com) y el servidor activo (`ollama serve`).

In [ ]:
# %pip install ollama langchain-ollama openai

import ollama
import json
import math
import time

# Verificar que Ollama está activo
try:
    modelos = ollama.list()
    nombres = [m['name'] for m in modelos.get('models', [])]
    print(f'✓ Ollama activo — modelos disponibles: {nombres or "(ninguno descargado)"}')
    print('Tip: ejecuta `ollama pull llama3.2` para descargar el modelo base de este notebook.')
except Exception as e:
    print(f'⚠️  No se pudo conectar con Ollama: {e}')
    print('Asegúrate de que el servidor esté activo: ollama serve')

MODELO = 'llama3.2'  # cambia aquí si tienes otro modelo

## 1. Primera llamada — generación básica

In [ ]:
respuesta = ollama.chat(
    model=MODELO,
    messages=[{'role': 'user', 'content': '¿Qué ventajas tienen los modelos de IA locales? Responde en 3 puntos.'}]
)
print(respuesta['message']['content'])

## 2. Streaming — tokens en tiempo real

In [ ]:
print('Respuesta en streaming:\n')
stream = ollama.chat(
    model=MODELO,
    messages=[{'role': 'user', 'content': 'Explica qué es RAG en 4 líneas.'}],
    stream=True
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)
print()  # salto de línea final

## 3. Conversación con historial

In [ ]:
def chat(historial: list, nueva_entrada: str, modelo: str = MODELO) -> str:
    """Añade mensaje y obtiene respuesta manteniendo el historial."""
    historial.append({'role': 'user', 'content': nueva_entrada})
    respuesta = ollama.chat(model=modelo, messages=historial)
    contenido = respuesta['message']['content']
    historial.append({'role': 'assistant', 'content': contenido})
    return contenido

historial = [{
    'role': 'system',
    'content': 'Eres un asistente experto en inteligencia artificial. Responde en español, de forma concisa.'
}]

r1 = chat(historial, 'Mi nombre es Carlos y estoy aprendiendo sobre agentes de IA.')
print('R1:', r1)

r2 = chat(historial, '¿Cuál fue el tema que mencioné y cómo me llamo?')
print('\nR2:', r2)

## 4. Medir velocidad — tokens por segundo

In [ ]:
def medir_velocidad(modelo: str, prompt: str) -> dict:
    """Mide el tiempo y la velocidad de generación de un modelo local."""
    inicio = time.time()
    tokens = 0
    texto = ''

    stream = ollama.chat(
        model=modelo,
        messages=[{'role': 'user', 'content': prompt}],
        stream=True
    )

    for chunk in stream:
        fragmento = chunk['message']['content']
        texto += fragmento
        tokens += len(fragmento.split())

    elapsed = time.time() - inicio
    return {
        'modelo': modelo,
        'tokens_aprox': tokens,
        'segundos': round(elapsed, 2),
        'tok_por_segundo': round(tokens / elapsed, 1),
        'texto': texto[:200] + '...'
    }

resultado = medir_velocidad(MODELO, 'Escribe un resumen de 100 palabras sobre los transformers en IA.')
print(f"Modelo:          {resultado['modelo']}")
print(f"Tokens generados: {resultado['tokens_aprox']}")
print(f"Tiempo:          {resultado['segundos']}s")
print(f"Velocidad:       {resultado['tok_por_segundo']} tok/s")
print(f"Texto:           {resultado['texto']}")

## 5. Salida estructurada en JSON

In [ ]:
def extraer_info(texto: str, esquema_desc: str) -> dict:
    """Extrae información estructurada de texto en formato JSON."""
    prompt = f"""Extrae la siguiente información del texto y devuelve JSON válido.
Campos a extraer: {esquema_desc}
Texto: {texto}
Responde SOLO con el JSON, sin texto adicional."""

    respuesta = ollama.chat(
        model=MODELO,
        messages=[{'role': 'user', 'content': prompt}],
        format='json'  # fuerza salida JSON
    )
    return json.loads(respuesta['message']['content'])

# Ejemplo: extraer datos de una reseña
resena = """El restaurante Casa Manolo en Barcelona es excelente. 
Los platos de marisco son increíbles, especialmente las gambas al ajillo. 
El precio es un poco elevado (unos 45€ por persona) pero vale la pena. 
El servicio podría mejorar, tardaron 20 minutos en atendernos."""

info = extraer_info(
    resena,
    'nombre_restaurante, ciudad, especialidad, precio_por_persona_euros, puntuacion_general_1_10, puntuacion_servicio_1_10'
)
print(json.dumps(info, indent=2, ensure_ascii=False))

## 6. Embeddings locales — búsqueda semántica sin internet

In [ ]:
# Requiere: ollama pull nomic-embed-text
MODELO_EMBED = 'nomic-embed-text'

def embedding(texto: str) -> list:
    try:
        r = ollama.embeddings(model=MODELO_EMBED, prompt=texto)
        return r['embedding']
    except Exception:
        # Fallback: si nomic no está disponible, simula con hash
        import hashlib
        h = hashlib.md5(texto.encode()).hexdigest()
        return [int(h[i:i+2], 16) / 255 for i in range(0, 32, 2)]

def similitud_coseno(a: list, b: list) -> float:
    dot = sum(x*y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(x**2 for x in b))
    return dot / (norm_a * norm_b + 1e-9)

# Base de conocimiento
documentos = [
    'LangGraph es un framework para construir agentes con grafos de estado y memoria persistente.',
    'Ollama permite ejecutar LLMs en local sin costes ni privacidad comprometida.',
    'RAG combina recuperación de documentos con generación para respuestas precisas.',
    'El fine-tuning con LoRA adapta un modelo base a un dominio específico con pocos recursos.',
    'El patrón ReAct alterna razonamiento y acción para resolver tareas complejas.',
]

print('Generando embeddings... (puede tardar si nomic no está cacheado)')
embs_docs = [embedding(d) for d in documentos]

def buscar(consulta: str, top_k: int = 2) -> list:
    emb_q = embedding(consulta)
    puntuaciones = [(similitud_coseno(emb_q, e), doc)
                   for e, doc in zip(embs_docs, documentos)]
    puntuaciones.sort(reverse=True)
    return puntuaciones[:top_k]

consulta = '¿Cómo ejecuto un modelo sin internet?'
print(f'\nConsulta: {consulta}')
for score, doc in buscar(consulta):
    print(f'  [{score:.3f}] {doc}')

## 7. API compatible con OpenAI — migración sin dolor

In [ ]:
from openai import OpenAI

# Punto clave: solo cambia base_url y api_key
client_local = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'  # cualquier string
)

# El resto del código es idéntico a openai
respuesta = client_local.chat.completions.create(
    model=MODELO,
    messages=[
        {'role': 'system', 'content': 'Eres un asistente técnico especializado en IA.'},
        {'role': 'user', 'content': '¿Cuáles son las ventajas de Q4_K_M frente a F16?'}
    ]
)
print(respuesta.choices[0].message.content)

## 8. Integración con LangChain

In [ ]:
# %pip install langchain-ollama
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_local = ChatOllama(model=MODELO, temperature=0.2)

# Cadena LCEL con modelo local
plantilla = ChatPromptTemplate.from_messages([
    ('system', 'Eres experto en {tema}. Responde de forma concisa en español.'),
    ('human', '{pregunta}')
])

cadena = plantilla | llm_local | StrOutputParser()

resultado = cadena.invoke({
    'tema': 'modelos de lenguaje',
    'pregunta': '¿Cuándo debo usar un modelo local en lugar de uno cloud?'
})
print(resultado)

## 9. Benchmark: local vs cloud — comparativa de coste

In [ ]:
# Estimación de costes para 1 millón de tokens
modelos_cloud = {
    'GPT-4o':              {'input': 2.50, 'output': 10.00},
    'Claude Sonnet 4.6':   {'input': 3.00, 'output': 15.00},
    'GPT-3.5-turbo':       {'input': 0.50, 'output': 1.50},
    'Gemini Flash 2.0':    {'input': 0.075, 'output': 0.30},
}

volumen_tokens = 10_000_000  # 10M tokens al mes
mix = {'input': 0.7, 'output': 0.3}  # 70% input, 30% output

print(f'Coste estimado para {volumen_tokens:,} tokens/mes\n')
print(f'{"Modelo":<25} {"Coste mensual (€)":>18}')
print('─' * 45)

for modelo, precios in modelos_cloud.items():
    tokens_in  = volumen_tokens * mix['input']
    tokens_out = volumen_tokens * mix['output']
    coste = (tokens_in / 1_000_000 * precios['input'] +
             tokens_out / 1_000_000 * precios['output'])
    print(f'{modelo:<25} {coste:>15.2f} €')

print('─' * 45)
print(f'{"Ollama (local)":<25} {0.0:>15.2f} €')
print(f'  (solo electricidad: ~{0.01 * 24 * 30:.1f} € electricidad/mes)')

## 10. Crear un asistente personalizado con Modelfile

In [ ]:
# Crear un Modelfile y registrar un asistente personalizado
modelfile_contenido = f"""FROM {MODELO}

PARAMETER temperature 0.2
PARAMETER num_ctx 4096

SYSTEM \"\"\"Eres un asistente especializado en inteligencia artificial y Python.
- Responde siempre en español
- Incluye ejemplos de código cuando sea relevante
- Sé conciso y directo
- Si no tienes certeza sobre algo, dilo claramente\"\"\"
"""

# Guardar Modelfile
with open('Modelfile.asistente-ia', 'w') as f:
    f.write(modelfile_contenido)

print('Modelfile creado. Para registrar el modelo:')
print('  ollama create asistente-ia -f Modelfile.asistente-ia')
print('  ollama run asistente-ia')

print('\nContenido del Modelfile:')
print(modelfile_contenido)

## 11. RAG completamente local — privacidad garantizada

In [ ]:
class RAGLocal:
    """Sistema RAG que funciona 100% en local: embeddings + LLM."""

    def __init__(self, modelo_llm: str = MODELO, modelo_embed: str = MODELO_EMBED):
        self.modelo_llm = modelo_llm
        self.modelo_embed = modelo_embed
        self.documentos: list[dict] = []

    def añadir_documento(self, texto: str, metadata: dict = None):
        emb = embedding(texto)
        self.documentos.append({'texto': texto, 'embedding': emb, 'metadata': metadata or {}})

    def recuperar(self, consulta: str, top_k: int = 3) -> list[dict]:
        emb_q = embedding(consulta)
        resultados = sorted(
            self.documentos,
            key=lambda d: similitud_coseno(emb_q, d['embedding']),
            reverse=True
        )
        return resultados[:top_k]

    def preguntar(self, consulta: str) -> str:
        docs = self.recuperar(consulta)
        contexto = '\n\n'.join(d['texto'] for d in docs)
        prompt = f"""Contexto (documentos recuperados):
{contexto}

Pregunta: {consulta}
Responde basándote en el contexto. Si la información no está disponible, indícalo."""
        respuesta = ollama.chat(
            model=self.modelo_llm,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return respuesta['message']['content']

# Construir base de conocimiento
rag = RAGLocal()
rag.añadir_documento('LangGraph permite construir agentes con grafos de estado, checkpointing y human-in-the-loop.')
rag.añadir_documento('Ollama ejecuta modelos como Llama 3.2, Mistral y Qwen en local sin enviar datos a servidores externos.')
rag.añadir_documento('La cuantización Q4_K_M reduce el tamaño del modelo al 25% con pérdida mínima de calidad.')
rag.añadir_documento('RAG es superior al fine-tuning para conocimiento que cambia frecuentemente o datos muy específicos.')

respuesta = rag.preguntar('¿Qué modelos puedo ejecutar con Ollama y cómo afecta la cuantización al rendimiento?')
print(respuesta)

---
**Tutorial relacionado:** [09-modelos-locales-ollama.md](../../llms/09-modelos-locales-ollama.md) · **Anterior:** [08-langgraph.ipynb](./08-langgraph.ipynb)